# Upload Benchmark Instances to Kaggle

Downloads 174 benchmark instances from PyVRP/Instances GitHub and uploads them as a Kaggle dataset (`elfateh/dana-instances`).

Instance counts: 28 cordeau + 56 solomon + 60 gehring + 30 x_instances = 174 total

In [ ]:
import os, subprocess, json, requests

BASE_URL = "https://raw.githubusercontent.com/PyVRP/Instances/main"
INSTANCE_DIR = "/kaggle/working/instances"
DATASET_DIR = "/kaggle/working/dana-instances"

SETS = {
    "cordeau": {
        "remote_dir": "MDVRPTW",
        "files": [f"PR{i}{s}.vrp" for i in range(11, 25) for s in ("A", "B")],
    },
    "solomon": {
        "remote_dir": "VRPTW/Solomon",
        "files": [
            "C101.vrp", "C102.vrp", "C103.vrp", "C104.vrp", "C105.vrp", "C106.vrp", "C107.vrp", "C108.vrp", "C109.vrp",
            "C201", "C202", "C203", "C204", "C205", "C206", "C207", "C208",
            "R101", "R102", "R103", "R104", "R105", "R106", "R107", "R108",
            "R109", "R110", "R111", "R112",
            "R201", "R202", "R203", "R204", "R205", "R206", "R207", "R208", "R209", "R210", "R211",
            "RC101", "RC102", "RC103", "RC104", "RC105", "RC106", "RC107", "RC108",
            "RC201", "RC202", "RC203", "RC204", "RC205", "RC206", "RC207", "RC208",
        ],
    },
    "gehring": {
        "remote_dir": "VRPTW",
        "files": [f"{t}{n}_10_{i}.vrp" for t in ("C", "R", "RC") for n in (1, 2) for i in range(1, 11)],
    },
    "x_instances": {
        "remote_dir": "CVRP",
        "files": [
            "X-n101-k25.vrp", "X-n106-k14.vrp", "X-n110-k13.vrp", "X-n115-k10.vrp",
            "X-n120-k6.vrp", "X-n125-k30.vrp", "X-n129-k18.vrp", "X-n134-k13.vrp",
            "X-n139-k10.vrp", "X-n143-k7.vrp", "X-n153-k22.vrp", "X-n157-k13.vrp",
            "X-n162-k11.vrp", "X-n167-k10.vrp", "X-n176-k26.vrp", "X-n181-k23.vrp",
            "X-n186-k15.vrp", "X-n190-k8.vrp", "X-n195-k51.vrp", "X-n200-k36.vrp",
            "X-n204-k19.vrp", "X-n209-k16.vrp", "X-n214-k11.vrp", "X-n219-k73.vrp",
            "X-n223-k34.vrp", "X-n228-k23.vrp", "X-n233-k16.vrp", "X-n237-k14.vrp",
            "X-n242-k48.vrp", "X-n247-k50.vrp",
        ],
    },
}

total_ok, total_fail = 0, 0
for set_name, info in SETS.items():
    dest = os.path.join(INSTANCE_DIR, set_name)
    os.makedirs(dest, exist_ok=True)
    ok, fail = 0, 0
    for fname in info["files"]:
        url = f"{BASE_URL}/{info['remote_dir']}/{fname}"
        path = os.path.join(dest, fname)
        if os.path.exists(path):
            ok += 1
            continue
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 200:
                with open(path, "w") as f:
                    f.write(r.text)
                ok += 1
            else:
                fail += 1
        except Exception:
            fail += 1
    print(f"{set_name}: {ok} ok, {fail} failed")
    total_ok += ok
    total_fail += fail

print(f"\nTotal: {total_ok} ok, {total_fail} failed")

In [ ]:
import shutil

os.makedirs(DATASET_DIR, exist_ok=True)
shutil.copytree(INSTANCE_DIR, DATASET_DIR, dirs_exist_ok=True)

with open(os.path.join(DATASET_DIR, "dataset-metadata.json"), "w") as f:
    json.dump({
        "title": "dana-instances",
        "id": "elfateh/dana-instances",
        "licenses": [{"name": "CC0-1.0"}],
    }, f)

# Always try version first (dataset already exists), fall back to create
r = subprocess.run(
    ["kaggle", "datasets", "version", "-p", DATASET_DIR, "-m", "Updated instances", "--dir-mode", "zip"],
    capture_output=True, text=True,
)
if r.returncode != 0:
    r2 = subprocess.run(
        ["kaggle", "datasets", "create", "-p", DATASET_DIR, "--dir-mode", "zip"],
        capture_output=True, text=True,
    )
    if r2.returncode != 0:
        print(f"Dataset upload failed: {r2.stderr.strip()}")
    else:
        print("Dataset created successfully.")
else:
    print("Dataset version updated successfully.")